<h2 style='color:blue' align='center'>Pneumonia Detection from Chest X-Rays using CNN</h2>

We're building the CNN model for the pneumonia detector — same dataset we already used for the ResNet50 and EfficientNet versions, so results stay comparable across all three.The idea here isn't to beat the transfer learning models. It's to have a lightweight CNN trained from scratch that's fast to run and still reasonably accurate, so the app has a "quick scan" option alongside the heavier models.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import os

<h4 style='color:purple'>Setting up the data paths</h4>

Kaggle's chest x-ray dataset comes pre-split into train/val/test folders, each with a NORMAL and PNEUMONIA subfolder. We'll point straight at those instead of doing our own split.

In [ ]:
train_dir = '/kaggle/input/chest-xray-pneumonia/chest_xray/train'
val_dir   = '/kaggle/input/chest-xray-pneumonia/chest_xray/val'
test_dir  = '/kaggle/input/chest-xray-pneumonia/chest_xray/test'

IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
for cls in ['NORMAL', 'PNEUMONIA']:
    count = len(os.listdir(os.path.join(train_dir, cls)))
    print(cls, '-', count, 'images')

The val folder in this dataset only has 16 images, which isn't really enough to track validation accuracy properly. We'll let Keras carve out a chunk of the training set instead and use the official val folder just as a sanity check later.

<h4 style='color:purple'>Data augmentation</h4>

X-ray datasets are usually small and a plain CNN will overfit on them fast. Augmenting the training images (small rotations, shifts, zoom) gives the model more variation to learn from without us needing more data.No horizontal flip here on purpose — flipping a chest x-ray left-to-right can flip which lung looks affected, which isn't something we want the model learning as "normal" variation.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    validation_split=0.15
)

test_datagen = ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

In [ ]:
train_generator.class_indices

class_indices should show `{'NORMAL': 0, 'PNEUMONIA': 1}` — worth checking this every time, since Keras assigns these alphabetically and it's easy to assume the wrong order when writing the prediction code later.

<h4 style='color:purple'>Taking a quick look at the images</h4>

In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(10, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i])
    plt.title('PNEUMONIA' if labels[i] == 1 else 'NORMAL')
    plt.axis('off')
plt.tight_layout()
plt.show()

<h4 style='color:purple'>Building the CNN</h4>

Keeping this fairly simple — three conv blocks, each one doubling the filter count, with batch norm after each conv to stabilize training and dropout before the dense layers to fight overfitting. Nothing fancy, just enough depth to actually learn something on a 224x224 input without taking forever to train.

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.summary()

Sigmoid + 1 output unit since this is binary (NORMAL vs PNEUMONIA), not multi-class — that's why we're not using softmax like in the CIFAR-10 notebook.

<h4 style='color:purple'>Compiling</h4>

Pneumonia detection is the kind of problem where missing an actual pneumonia case (false negative) is worse than flagging a healthy scan for review (false positive). So besides accuracy we're also tracking precision and recall during training, not just at the end.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)

<h4 style='color:purple'>Callbacks</h4>

Two callbacks doing the heavy lifting here — early stopping so training doesn't keep going once val_loss stops improving, and a checkpoint that only saves the best version of the model instead of whatever the last epoch happened to land on.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        'cnn_v1.keras',
        monitor='val_loss',
        save_best_only=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

<h4 style='color:purple'>Handling class imbalance</h4>

This dataset has way more PNEUMONIA images than NORMAL ones, so without correcting for that the model would just learn to lean towards predicting pneumonia and still score decent accuracy. class_weight tells Keras to penalize mistakes on the minority class more.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

labels_array = train_generator.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_array),
    y=labels_array
)
class_weights = dict(enumerate(class_weights))
class_weights

<h4 style='color:purple'>Training</h4>

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks
)

<h4 style='color:purple'>Plotting accuracy and loss</h4>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(history.history['accuracy'], label='train')
ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy')
ax[0].set_xlabel('Epoch')
ax[0].legend()

ax[1].plot(history.history['loss'], label='train')
ax[1].plot(history.history['val_loss'], label='val')
ax[1].set_title('Loss')
ax[1].set_xlabel('Epoch')
ax[1].legend()

plt.tight_layout()
plt.show()

<h4 style='color:purple'>Evaluating on the test set</h4>

In [ ]:
test_loss, test_acc, test_precision, test_recall = model.evaluate(test_generator)
print(f'Test accuracy:  {test_acc:.4f}')
print(f'Test precision: {test_precision:.4f}')
print(f'Test recall:    {test_recall:.4f}')

In [ ]:
y_pred_probs = model.predict(test_generator)
y_pred = (y_pred_probs > 0.5).astype(int).reshape(-1)
y_true = test_generator.classes

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL', 'PNEUMONIA'],
            yticklabels=['NORMAL', 'PNEUMONIA'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - CNN')
plt.show()

<h4 style='color:purple'>Checking a few predictions</h4>

In [ ]:
test_images, test_labels = next(test_generator)
preds = model.predict(test_images)

plt.figure(figsize=(12, 8))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(test_images[i])
    actual = 'PNEUMONIA' if test_labels[i] == 1 else 'NORMAL'
    predicted = 'PNEUMONIA' if preds[i] > 0.5 else 'NORMAL'
    color = 'green' if actual == predicted else 'red'
    plt.title(f'Actual: {actual}\nPred: {predicted} ({preds[i][0]:.2f})', color=color, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

<h4 style='color:purple'>Saving the model</h4>

ModelCheckpoint already saved the best version during training as `cnn_v1.keras`, so this is mostly a final explicit save in case the run gets interrupted or you want to push a copy somewhere else after evaluating.

In [ ]:
model.save('cnn_v1.keras')

Quick sanity check before downloading — reload it fresh and make sure it's actually the CNN and not some leftover variable from another cell. Worth doing every time, this exact mix-up is what broke the previous version of this model.

In [ ]:
check_model = keras.models.load_model('cnn_v1.keras')
check_model.summary()

Once this looks right, download `cnn_v1.keras` and drop it into the `model/` folder, replacing the old `cnn_v1.h5`. The FastAPI backend already expects this filename in `MODEL_PATHS`, so no other code changes needed on that side.